# Day 4 baseline 建模

本 notebook 用于跑通 Scania APS 预测性维护项目的第一版 baseline 建模闭环。Day 4 只比较基础缺失处理策略和 baseline 模型，不做 XGBoost、不做 GridSearch、不做阈值遍历，也不进行风险分层。

## 1. 导入依赖

本阶段统一从 `config/config.yaml` 读取路径、标签映射、缺失值 token、业务成本和默认模型参数。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import pandas as pd

from scania_aps.config import get_config
from scania_aps.data.load_data import load_train_test_with_target
from scania_aps.models.train_baseline import train_and_evaluate_baselines

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## 2. 读取 cfg

成本参数、默认阈值和高缺失字段阈值都来自配置文件，避免在 notebook 中重复写死。

In [2]:
cfg = get_config(PROJECT_ROOT / "config" / "config.yaml")

cfg.train_raw, cfg.test_raw, cfg.default_threshold, cfg.high_missing_threshold, cfg.false_positive_cost, cfg.false_negative_cost

(WindowsPath('C:/Scania APS/data/raw/aps_failure_training_set.csv'),
 WindowsPath('C:/Scania APS/data/raw/aps_failure_test_set.csv'),
 0.5,
 0.8,
 10,
 500)

## 3. 读取 train/test 并映射 target

只读取官方 train/test，不合并后重新划分。`class` 映射规则来自 cfg。

In [3]:
train_df, test_df = load_train_test_with_target(cfg)

dataset_shape = pd.DataFrame(
    [
        {"dataset": "train", "rows": train_df.shape[0], "columns": train_df.shape[1]},
        {"dataset": "test", "rows": test_df.shape[0], "columns": test_df.shape[1]},
    ]
)
dataset_shape

,dataset,rows,columns
0,train,60000,172
1,test,16000,172


## 4. 标签分布检查

Day 4 继续关注类别不平衡问题，因此后续指标表不使用 accuracy 作为核心指标。

In [4]:
label_distribution = pd.concat(
    [
        train_df[cfg.label_column].value_counts(normalize=False).rename("count").to_frame().assign(dataset="train"),
        test_df[cfg.label_column].value_counts(normalize=False).rename("count").to_frame().assign(dataset="test"),
    ]
).reset_index(names="class")

label_distribution["ratio"] = label_distribution.groupby("dataset")["count"].transform(lambda x: x / x.sum())
label_distribution[["dataset", "class", "count", "ratio"]]

,dataset,class,count,ratio
0,train,neg,59000,0.983333
1,train,pos,1000,0.016667
2,test,neg,15625,0.976562
3,test,pos,375,0.023438


## 5. 定义 baseline 缺失处理策略

`median_all` 保留全部字段并使用训练集中位数填充；`drop_high_missing_median` 只基于训练集识别高缺失字段，然后在 train/test 同步删除这些字段，再使用训练集中位数填充。

In [5]:
strategies = ["median_all", "drop_high_missing_median"]
strategies

['median_all', 'drop_high_missing_median']

## 6. 训练 Dummy baseline 和 Logistic Regression baseline

这里使用配置中的默认阈值，不做阈值遍历。

In [6]:
metrics_df, predictions_df = train_and_evaluate_baselines(
    train_df=train_df,
    test_df=test_df,
    cfg=cfg,
    strategies=strategies,
)

metrics_df

,model_name,strategy,threshold,precision,recall,f1,f2,average_precision,tn,fp,fn,tp,false_positive_cost,false_negative_cost,total_cost,dataset,n_features,n_dropped_features
0,dummy_prior,median_all,0.5,0.000000,0.000000,0.000000,0.000000,0.023438,15625,0,375,0,10,500,187500,test,170,0
1,logistic_regression_balanced,median_all,0.5,0.481894,0.922667,0.633120,0.779982,0.798196,15253,372,29,346,10,500,18220,test,170,0
2,dummy_prior,drop_high_missing_median,0.5,0.000000,0.000000,0.000000,0.000000,0.023438,15625,0,375,0,10,500,187500,test,168,2
3,logistic_regression_balanced,drop_high_missing_median,0.5,0.486034,0.928000,0.637947,0.785199,0.799356,15257,368,27,348,10,500,17180,test,168,2


## 7. 查看核心指标

重点观察 precision、recall、F2、PR-AUC 和 total cost。Dummy baseline 用于提醒我们：在极不平衡数据中，只预测多数类会漏掉全部正类。

In [7]:
metrics_view = metrics_df[
    [
        "model_name",
        "strategy",
        "threshold",
        "precision",
        "recall",
        "f1",
        "f2",
        "average_precision",
        "fp",
        "fn",
        "total_cost",
        "n_features",
        "n_dropped_features",
    ]
].sort_values("total_cost")

metrics_view

,model_name,strategy,threshold,precision,recall,f1,f2,average_precision,fp,fn,total_cost,n_features,n_dropped_features
3,logistic_regression_balanced,drop_high_missing_median,0.5,0.486034,0.928000,0.637947,0.785199,0.799356,368,27,17180,168,2
1,logistic_regression_balanced,median_all,0.5,0.481894,0.922667,0.633120,0.779982,0.798196,372,29,18220,170,0
0,dummy_prior,median_all,0.5,0.000000,0.000000,0.000000,0.000000,0.023438,0,375,187500,170,0
2,dummy_prior,drop_high_missing_median,0.5,0.000000,0.000000,0.000000,0.000000,0.023438,0,375,187500,168,2


## 8. 保存 Day 4 输出

输出文件保存在 `outputs/metrics/` 和 `outputs/predictions/`，这些是本地运行产物，默认不提交 GitHub。

In [8]:
cfg.metrics_dir.mkdir(parents=True, exist_ok=True)
cfg.predictions_dir.mkdir(parents=True, exist_ok=True)

metrics_path = cfg.metrics_dir / "day4_baseline_metrics.csv"
predictions_path = cfg.predictions_dir / "day4_baseline_predictions.csv"

metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
predictions_df.to_csv(predictions_path, index=False, encoding="utf-8-sig")

metrics_path, predictions_path

(WindowsPath('C:/Scania APS/outputs/metrics/day4_baseline_metrics.csv'),
 WindowsPath('C:/Scania APS/outputs/predictions/day4_baseline_predictions.csv'))

## 9. Day 4 小结

- Day 4 已跑通第一版 baseline 建模闭环。
- 当前只比较基础缺失处理策略和 baseline 模型，不代表最终模型效果。
- Logistic Regression baseline 可以作为 Day 5 提升模型的参照。
- 后续应继续围绕 recall、F2、PR-AUC 和 total cost 评估模型，而不是使用 accuracy 作为核心目标。